# Projet : Analyse des Offres d'Emploi LinkedIn avec Snowflake

Dans ce projet nous allons exploiter un ensemble de données provenant de Linkedln afin d'effectuer des analyses pertinentes sur le marché de l'emploi. Pour ce faire nous allons travailler sur Snowflake et streamlit avec les fichiers du bucket S3 :  s3://snowflake-lab-bucket/

Dans ce notebook, vous pourrez retrouver les codes avec les explications de ceux-ci pour chaque étape bien que également transmit sur GitHub comme indiqué dans l'enoncée. 
Notre travail se compose en trois grandes parties : 'I-Etapes d'initiation de linkedln', 'II - Analyse des données' et 'III - Problèmes rencontrés et solutions apportées' avec une visualisation sur streamlit via le lien : https://app.snowflake.com/streamlit/lhzicag/cxb46557/#/apps/4tl6dhmaprcaaumew4gk

# I - Etapes d'initiation de linkedln

## Première étape : Création de la base de données

Pour ce faire, nous avons utilisé un code SQL afin de créer la base nommée 'linkedln' : 

In [ ]:
CREATE DATABASE IF NOT EXISTS linkedin;

Ensuite, nous avons créé un schéma type nommé 'raw' pour organiser les tables et les objets dans une base et nous avons affiché notre base précédement faite avec le code suivant : 

In [ ]:
USE DATABASE linkedin;
CREATE SCHEMA IF NOT EXISTS raw;
SHOW DATABASES LIKE 'linkedin';

## Seconde étape: Création d'un stage externe

Nous avons ainsi créé un stage pointant vers le bucket sur lequel nous allons travailler pour éxtrère les données et ainsi travailler dessus. 

In [ ]:
USE DATABASE linkedin;
USE SCHEMA raw;
CREATE OR REPLACE STAGE s3_linkedin_stage
    URL = 's3://snowflake-lab-bucket/'
    FILE_FORMAT = (TYPE = 'CSV');

LIST @s3_linkedin_stage;

## Troisième étape: Définition des formats de fichiers

Pour définir le formats des fichiers CSV, nous avons utilisé la commade SQL suivante : 

In [ ]:
CREATE OR REPLACE FILE FORMAT linkedin.raw.csv_format
  TYPE = 'CSV'
  FIELD_DELIMITER = ','
  RECORD_DELIMITER = '\n'
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  NULL_IF = ('NULL', 'null', '', 'N/A')
  EMPTY_FIELD_AS_NULL = TRUE;

Puis, pour défini pour les fichiers JSON, nous avons utilisé : 

In [ ]:
CREATE OR REPLACE FILE FORMAT linkedin.raw.json_format
  TYPE = 'JSON'
  STRIP_OUTER_ARRAY = TRUE
  NULL_IF = ('NULL', 'null');

## Quatrième étape: Création des tables 

A l'aide des différents fichier transmit dans 's3://snowflake-lab-bucket/' nous avons réalisé les tables dans la base de donnée 'linkedln'. Nous avons ainsi divisé chaque fichiers en table avec les valeurs correspondants aux colonnes des Excels (pour les fichiers CSV) ou les clefs/valeurs (pour les fichiers JSON). 

In [ ]:
USE DATABASE linkedin;
USE SCHEMA raw;

-- 1. job_postings
CREATE OR REPLACE TABLE linkedin.raw.job_postings (
  job_id                     VARCHAR,
  company_name               VARCHAR,
  title                      VARCHAR,
  description                TEXT,
  max_salary                 VARCHAR,
  med_salary                 VARCHAR,
  min_salary                 VARCHAR,
  pay_period                 VARCHAR,
  formatted_work_type        VARCHAR,
  location                   VARCHAR,
  applies                    VARCHAR,
  original_listed_time       VARCHAR,
  remote_allowed             VARCHAR,
  views                      VARCHAR,
  job_posting_url            VARCHAR,
  application_url            VARCHAR,
  application_type           VARCHAR,
  expiry                     VARCHAR,
  closed_time                VARCHAR,
  formatted_experience_level VARCHAR,
  skills_desc                TEXT,
  listed_time                VARCHAR,
  posting_domain             VARCHAR,
  sponsored                  VARCHAR,
  work_type                  VARCHAR,
  currency                   VARCHAR,
  compensation_type          VARCHAR
);

-- 2. benefits
CREATE OR REPLACE TABLE benefits (
    job_id      VARCHAR(50),
    inferred    BOOLEAN,
    type        VARCHAR(100)
);

-- 3. companies
CREATE OR REPLACE TABLE companies (
    company_id      VARCHAR(50),
    name            VARCHAR(255),
    description     TEXT,
    company_size    INT,
    state           VARCHAR(100),
    country         VARCHAR(100),
    city            VARCHAR(100),
    zip_code        VARCHAR(20),
    address         VARCHAR(500),
    url             VARCHAR(500)
);

-- 4. employee_counts
CREATE OR REPLACE TABLE linkedin.raw.employee_counts (
  company_id     VARCHAR,
  employee_count VARCHAR,
  follower_count VARCHAR,
  time_recorded  VARCHAR
);

-- 5. job_skills
CREATE OR REPLACE TABLE job_skills (
    job_id      VARCHAR(50),
    skill_abr   VARCHAR(50)
);

-- 6. job_industries
CREATE OR REPLACE TABLE job_industries (
    job_id          VARCHAR(50),
    industry_id     VARCHAR(50)
);

-- 7. company_specialities
CREATE OR REPLACE TABLE company_specialities (
    company_id  VARCHAR(50),
    speciality  VARCHAR(255)
);

-- 8. company_industries
CREATE OR REPLACE TABLE company_industries (
    company_id  VARCHAR(50),
    industry    VARCHAR(255)
);

Enfin pour visualiser le travail sur notre base de donnée et voir les tables, nous rentrons la commande : 

In [ ]:
SHOW TABLES IN SCHEMA linkedin.raw;

## Cinquième étape: Chargement des données 

A l'aide d'un programme SQL, nous avons intégrer les données des fichiers dans nos tables précédemment créé

In [ ]:
SHOW TABLES IN SCHEMA linkedin.raw;
USE SCHEMA linkedin.raw;
 
COPY INTO linkedin.raw.job_postings
FROM @linkedin.public.s3_linkedin_stage/job_postings.csv
FILE_FORMAT = (FORMAT_NAME = 'linkedin.public.csv_format')
ON_ERROR = 'CONTINUE';
 
COPY INTO linkedin.raw.benefits
FROM @linkedin.public.s3_linkedin_stage/benefits.csv
FILE_FORMAT = (FORMAT_NAME = 'linkedin.public.csv_format')
ON_ERROR = 'CONTINUE';
 
COPY INTO linkedin.raw.employee_counts
FROM @linkedin.public.s3_linkedin_stage/employee_counts.csv
FILE_FORMAT = (FORMAT_NAME = 'linkedin.public.csv_format')
ON_ERROR = 'CONTINUE';
 
COPY INTO linkedin.raw.job_skills
FROM @linkedin.public.s3_linkedin_stage/job_skills.csv
FILE_FORMAT = (FORMAT_NAME = 'linkedin.public.csv_format')
ON_ERROR = 'CONTINUE';
 
COPY INTO linkedin.raw.companies
FROM (
  SELECT
    $1:company_id::NUMBER   AS company_id,
    $1:name::VARCHAR        AS name,
    $1:description::TEXT    AS description,
    $1:company_size::NUMBER AS company_size,
    $1:state::VARCHAR       AS state,
    $1:country::VARCHAR     AS country,
    $1:city::VARCHAR        AS city,
    $1:zip_code::VARCHAR    AS zip_code,
    $1:address::VARCHAR     AS address,
    $1:url::VARCHAR         AS url
  FROM @linkedin.public.s3_linkedin_stage/companies.json
  (FILE_FORMAT => 'linkedin.public.json_format')
)
ON_ERROR = 'CONTINUE';
 
 
COPY INTO linkedin.raw.job_industries
FROM (
  SELECT
    $1:job_id::NUMBER      AS job_id,
    $1:industry_id::NUMBER AS industry_id
  FROM @linkedin.public.s3_linkedin_stage/job_industries.json
  (FILE_FORMAT => 'linkedin.public.json_format')
)
ON_ERROR = 'CONTINUE';
 
COPY INTO linkedin.raw.company_specialities
FROM (
  SELECT
    $1:company_id::NUMBER  AS company_id,
    $1:speciality::VARCHAR AS speciality
  FROM @linkedin.public.s3_linkedin_stage/company_specialities.json
  (FILE_FORMAT => 'linkedin.public.json_format')
)
ON_ERROR = 'CONTINUE';
 
COPY INTO linkedin.raw.company_industries
FROM (
  SELECT
    $1:company_id::NUMBER AS company_id,
    $1:industry::VARCHAR  AS industry
  FROM @linkedin.public.s3_linkedin_stage/company_industries.json
  (FILE_FORMAT => 'linkedin.public.json_format')
)
ON_ERROR = 'CONTINUE';

Finalement nous avons effectué le code SQL suivant pour vérifier l'import : 

In [ ]:
USE SCHEMA linkedin.raw;
 
SELECT 'job_postings'         AS table_name, COUNT(*) AS nb_lignes FROM linkedin.raw.job_postings        UNION ALL
SELECT 'benefits'             AS table_name, COUNT(*) AS nb_lignes FROM linkedin.raw.benefits            UNION ALL
SELECT 'companies'            AS table_name, COUNT(*) AS nb_lignes FROM linkedin.raw.companies           UNION ALL
SELECT 'employee_counts'      AS table_name, COUNT(*) AS nb_lignes FROM linkedin.raw.employee_counts     UNION ALL
SELECT 'job_skills'           AS table_name, COUNT(*) AS nb_lignes FROM linkedin.raw.job_skills          UNION ALL
SELECT 'job_industries'       AS table_name, COUNT(*) AS nb_lignes FROM linkedin.raw.job_industries      UNION ALL
SELECT 'company_specialities' AS table_name, COUNT(*) AS nb_lignes FROM linkedin.raw.company_specialities UNION ALL
SELECT 'company_industries'   AS table_name, COUNT(*) AS nb_lignes FROM linkedin.raw.company_industries;

## Sixième étape : Transformations nécessaires 

=> Après implémentation des données, nous avons vue que certaine modification était nécessaire pour que les requêtes futures soient comprisent par streamlit et que les résultats ne soient pas faussées. Nous faisons ainsi des normalisations et standardisatiosn. 

### 1) Dates

Premièrement, nous avons dû convertir les timestamps en dates lisibles pour nous. En effet, dans nos fichiers CSV, les dates sont stockées sous forme brut ('Timestamps Unix' correspondant au nombres de secondes depuis la date inscrite). Nous avons ainsi crée une nouvelle colonne 'time_recorded_date' pour afficher la date en année/mois/jour.

In [ ]:
UPDATE linkedin.raw.employee_counts
SET time_recorded_date = TO_TIMESTAMP(time_recorded::NUMBER)
WHERE time_recorded IS NOT NULL;
 
SELECT time_recorded, time_recorded_date 
FROM linkedin.raw.employee_counts 
LIMIT 5;

### 2) Télétravail

En second lieu, nous avons normalisé la colonne du télétravail en forme booléen. En effet, la colonne dans le fichier CSV est remplie avec des valeurs incohérentes qui diffères d'une ligne à une autre bien que voulant dire la même chose (ex: télétrvail peut être inscrit par un '1', un '1.0' ou même un 'True'). C'est pourquoi, nous avons toutsimplifié en ne mettant que des 'True' ou 'False' pour ainsi corriger les erreurs lors d'executions de requêtes. 

In [ ]:
ALTER TABLE linkedin.raw.job_postings
  ADD COLUMN is_remote BOOLEAN;
 
UPDATE linkedin.raw.job_postings
SET is_remote = CASE
  WHEN remote_allowed IN ('1', '1.0', 'True', 'true') THEN TRUE
  ELSE FALSE
END;
-- Vérification
SELECT remote_allowed, is_remote, COUNT(*) FROM linkedin.raw.job_postings
GROUP BY remote_allowed, is_remote;

### 3) Salaires

Troisièmement, nous avons changé les salaires écrits en textes, en nombre. En effet, en premier lieu, nous avions implémenté les valeurs de salaires en texte (VARCHAR), cependant pour les calculs futurs (ex: moyennes) nous avons besoin de vrais nombres. 

In [ ]:
ALTER TABLE linkedin.raw.job_postings ADD COLUMN max_salary_num FLOAT;
ALTER TABLE linkedin.raw.job_postings ADD COLUMN min_salary_num FLOAT;
ALTER TABLE linkedin.raw.job_postings ADD COLUMN med_salary_num FLOAT;
 
UPDATE linkedin.raw.job_postings
SET
  max_salary_num = TRY_TO_DOUBLE(max_salary),
  min_salary_num = TRY_TO_DOUBLE(min_salary),
  med_salary_num = TRY_TO_DOUBLE(med_salary);

Pour vérifier l'import, nous avons effectué la commande SQL suivante : 

In [ ]:
SELECT max_salary, max_salary_num FROM linkedin.raw.job_postings
WHERE max_salary IS NOT NULL LIMIT 5;

### 4) Fiabilité

La dernière étape de 'modification' à été de vérifier les diférentes valeurs 'NULL'des tables que nous allons par la suite exploiter dans la seconde partie du projet. En effet, la table job_postings comportents de nombreuses offres sans nom d'entreprise et n'ayant aucun montant d'inscrit pour certaines cases de la colonne des salaires. 

In [ ]:
-- 4.Vérifier les valeurs NULL dans la table job_postings, job_industries et companies car ces tables vont être utiles pour les analyses 

-- Job_postings :
SELECT
  COUNT(*)                                                    AS total,
  SUM(CASE WHEN title IS NULL THEN 1 ELSE 0 END)             AS null_title,
  SUM(CASE WHEN company_name IS NULL THEN 1 ELSE 0 END)      AS null_company,
  SUM(CASE WHEN location IS NULL THEN 1 ELSE 0 END)          AS null_location,
  SUM(CASE WHEN max_salary_num IS NULL THEN 1 ELSE 0 END)    AS null_salary,
  SUM(CASE WHEN formatted_work_type IS NULL THEN 1 ELSE 0 END) AS null_work_type
FROM linkedin.raw.job_postings;
 
-- Job_industries:
SELECT
  COUNT(*)                                                    AS total,
  SUM(CASE WHEN job_id IS NULL THEN 1 ELSE 0 END)            AS null_job_id,
  SUM(CASE WHEN industry_id IS NULL THEN 1 ELSE 0 END)       AS null_industry_id
FROM linkedin.raw.job_industries;
 
-- Companies :
SELECT
  COUNT(*)                                                    AS total,
  SUM(CASE WHEN company_id IS NULL THEN 1 ELSE 0 END)        AS null_company_id,
  SUM(CASE WHEN name IS NULL THEN 1 ELSE 0 END)              AS null_name,
  SUM(CASE WHEN company_size IS NULL THEN 1 ELSE 0 END)      AS null_company_size,
  SUM(CASE WHEN country IS NULL THEN 1 ELSE 0 END)           AS null_country
FROM linkedin.raw.companies;

# II - Analyse des données 

Dans cette seconde grande étape de notre projet, nous avons analysées différents point de notre base de données Linkedln et nous les avons visualisé à l'aide de streamlit. Nous pouvons appercevoir le rendue visuel via le lien suivant : https://app.snowflake.com/streamlit/lhzicag/cxb46557/#/apps/4tl6dhmaprcaaumew4gk

## 1) Première analyse :

### Top 10 des titres de postes les plus publiés par industrie

Pou répondre à cette problématique, nous avons créé le code python suivant que nous avons implémenter directement dans streamlit retrouvable sur GitHub :

In [ ]:
## Code mit dans Streamlit

import streamlit as st
import pandas as pd
 
st.title("🏆 Top 10 des titres de postes les plus publiés par industrie")
 
conn = st.connection("snowflake")
session = conn.session()
 
queried_data = session.sql("""
    SELECT 
        ji.industry_id AS industry,
        jp.title,
        COUNT(*) AS nb_offres
    FROM linkedin.raw.job_postings jp
    JOIN linkedin.raw.job_industries ji ON jp.job_id = ji.job_id
    WHERE jp.title IS NOT NULL
    GROUP BY ji.industry_id, jp.title
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ji.industry_id ORDER BY COUNT(*) DESC) <= 10
    ORDER BY ji.industry_id, nb_offres DESC
""")
 
df = queried_data.to_pandas()
 
industries = df["INDUSTRY"].unique().tolist()
industrie_choisie = st.selectbox("Choisissez une industrie :", industries)
 
df_filtre = df[df["INDUSTRY"] == industrie_choisie].reset_index(drop=True)
 
st.subheader(f"Top 10 pour : {industrie_choisie}")
st.bar_chart(data=df_filtre, x="TITLE", y="NB_OFFRES")
 
st.subheader("Données détaillées")
st.dataframe(df_filtre[["TITLE", "NB_OFFRES"]])

Nous obtenons ainsi le visuel suivant : 

![Top 10 des titres de postes les plus publiés par industrie](https://raw.githubusercontent.com/clara-cj/Projet-Linkedln-EmmaNEDELEC-Clara-JULIEN/8366e84d2c1aed200fa5165174c04939cfe2e8d5/Images/Streamlit-1er%20analyses.png)



![Top 10 des titres de postes les plus publiés par industrie](https://raw.githubusercontent.com/clara-cj/Projet-Linkedln-EmmaNEDELEC-Clara-JULIEN/8366e84d2c1aed200fa5165174c04939cfe2e8d5/Images/Streamlit-1er%20analyses%20details.png)

=> Nous pouvons observer sur cet interface les offres d'emploi publiés par industrie, le tableau, ainsi que le graphe, nous donne les postes les plus fréquemments publiés avec leurs nombres d'offres pour une entreprise pré-définit. Ici, nous avons prit en exemple l'entreprise avec l'ID n°144 qui propose à titre d'exemple 4 offres d'emploies en tant que 'Senior Electrical Engineer'.

## 2) Deuxième analyse :

### Top 10 des postes les mieux rémunérés par industrie

De la même manière que pour la première analyse nous avons inscrit le code python suivant : 

In [ ]:
import streamlit as st
import pandas as pd

st.title("💰 Top 10 des postes les mieux rémunérés par industrie")

conn = st.connection("snowflake")
session = conn.session()

queried_data = session.sql("""
    SELECT
        ji.industry_id AS industry,
        jp.title,
        ROUND(AVG(jp.med_salary), 0) AS salaire_median_moyen
    FROM linkedin.raw.job_postings jp
    JOIN linkedin.raw.job_industries ji ON jp.job_id = ji.job_id
    WHERE jp.title IS NOT NULL
      AND jp.med_salary IS NOT NULL
      AND jp.pay_period = 'YEARLY'
    GROUP BY ji.industry_id, jp.title
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ji.industry_id ORDER BY AVG(jp.med_salary) DESC) <= 10
    ORDER BY ji.industry_id, salaire_median_moyen DESC
""")

df = queried_data.to_pandas()

industries = df["INDUSTRY"].unique().tolist()
industrie_choisie = st.selectbox("Choisissez une industrie :", industries)

df_filtre = df[df["INDUSTRY"] == industrie_choisie].reset_index(drop=True)

st.subheader(f"Top 10 pour : {industrie_choisie}")
st.bar_chart(data=df_filtre, x="TITLE", y="SALAIRE_MEDIAN_MOYEN")

st.subheader("Données détaillées")
st.dataframe(df_filtre[["TITLE", "SALAIRE_MEDIAN_MOYEN"]])

Nous obtenons ainsi le visuel suivant : 

![Top 10 des titres de postes les plus publiés par industrie](https://raw.githubusercontent.com/clara-cj/Projet-Linkedln-EmmaNEDELEC-Clara-JULIEN/8366e84d2c1aed200fa5165174c04939cfe2e8d5/Images/Streamlit-2em%20analyses.png)

![Top 10 des titres de postes les plus publiés par industrie](https://raw.githubusercontent.com/clara-cj/Projet-Linkedln-EmmaNEDELEC-Clara-JULIEN/8366e84d2c1aed200fa5165174c04939cfe2e8d5/Images/Streamlit-2em%20analyses%20details.png)

=> Les métiers les plus rémunérés de l'industrie 104 sont de loin Plant Manager (Directeur d'usine) et Controller (Contrôleur de Gestion) avec respectivement un salaire de 180000 et 160000. En revanche le poste de Field Service Engineer ne reçoit qu'un salaire de 11000. Ainsi on remarque que les postes de direction reçoivent un salaire bien supérieur à ceux qui exercent des fonctions administratives ou de support.

## 3) Troisieme analyse : 

### Répartition des offres d’emploi par taille d’entreprise

Pour cette analyse nous avons effectué ce code python sur streamlit : 

In [ ]:
import streamlit as st
 
conn = st.connection("snowflake")
session = conn.session()
 
st.title("🏢 Répartition des offres d'emploi par taille d'entreprise")
 
queried_data = session.sql("""
    SELECT 
        CASE c.company_size
            WHEN 0 THEN '0 - 1 employé'
            WHEN 1 THEN '1 - 10 employés'
            WHEN 2 THEN '11 - 50 employés'
            WHEN 3 THEN '51 - 200 employés'
            WHEN 4 THEN '201 - 500 employés'
            WHEN 5 THEN '501 - 1000 employés'
            WHEN 6 THEN '1001 - 5000 employés'
            WHEN 7 THEN '5000+ employés'
            ELSE 'Non renseigné'
        END AS taille_entreprise,
        COUNT(*) AS nb_offres
    FROM linkedin.raw.job_postings jp
    JOIN linkedin.raw.companies c 
        ON TRY_TO_NUMBER(jp.company_name) = c.company_id
    WHERE c.company_size IS NOT NULL
    GROUP BY c.company_size, taille_entreprise
    ORDER BY c.company_size
""")
 
df = queried_data.to_pandas()
 
st.subheader("Graphique")
st.bar_chart(data=df, x="TAILLE_ENTREPRISE", y="NB_OFFRES")
 
st.subheader("Données détaillées")
st.dataframe(df[["TAILLE_ENTREPRISE", "NB_OFFRES"]])

Nous obtenons ainsi le visuel suivant :

![Top 10 postes les mieux rémunérés](https://raw.githubusercontent.com/clara-cj/Projet-Linkedln-EmmaNEDELEC-Clara-JULIEN/main/Images/Streamlit-3em%20analyses.png)

![Top 10 postes les mieux rémunérés - détails](https://raw.githubusercontent.com/clara-cj/Projet-Linkedln-EmmaNEDELEC-Clara-JULIEN/main/Images/Streamlit-3em%20analyses%20details.png)

=> Lors de cette analyse, on observe que les grandes structures sont celles qui proposent le plus d'offres (plus de 5000). Ainsi même si chaque structure propose un grand nombre d'offres, ce sont les plus grosses entreprises qui proposent le plus d'offres.

## 4) Quatrieme analyse : 

### Répartition des offres d’emploi par secteur d’activité

Pour cette analyse nous avons utilisé la commande suivante sur streamlit : 

In [ ]:
import streamlit as st

conn = st.connection("snowflake")
session = conn.session()

st.title("Analyse 4 : 🏭 Répartition des offres d'emploi par secteur d'activité")
 
queried_data = session.sql("""
    SELECT
        ci.industry AS secteur,
        COUNT(*) AS nb_offres
    FROM linkedin.raw.job_postings jp
    JOIN linkedin.raw.companies c 
        ON TRY_TO_NUMBER(jp.company_name) = c.company_id
    JOIN linkedin.raw.company_industries ci 
        ON c.company_id = ci.company_id
    WHERE ci.industry IS NOT NULL
    GROUP BY ci.industry
    ORDER BY nb_offres DESC
""")
 
df = queried_data.to_pandas()
 
st.subheader("Graphique")
st.bar_chart(data=df, x="SECTEUR", y="NB_OFFRES")
 
st.subheader("Données détaillées")
st.dataframe(df[["SECTEUR", "NB_OFFRES"]])

Nous obtenons ainsi le visuel suivant : 

![Répartition par taille d'entreprise](https://raw.githubusercontent.com/clara-cj/Projet-Linkedln-EmmaNEDELEC-Clara-JULIEN/main/Images/Streamlit-4em%20analyses.png)

![Répartition par taille d'entreprise - détails](https://raw.githubusercontent.com/clara-cj/Projet-Linkedln-EmmaNEDELEC-Clara-JULIEN/main/Images/Streamlit-4em%20analyses%20details.png)

=> On observe sur ce graphique que les secteurs qui ressortent le plus sont celui de l'informatique et du numérique avec plus de 38000 offres, puis on retrouve les agences de recrutement avec plus de 35000 offres et le domaine de la vente propose plus de 27000 offres. On remarque également que si on associe Informatique, Internet et Logiciels, on retrouve plus de 77000 offres. Ainsi le monde de le Tech est le secteur d'activité le plus fort.

## 5) Cinquième analyse 

### Répartition des offres d’emploi par type d’emploi (temps plein, stage, temps partiel)

In [ ]:
import streamlit as st

conn = st.connection("snowflake")
session = conn.session()
 
st.title("Analyse 5 : 💼 Répartition des offres d'emploi par type d'emploi")
 
queried_data = session.sql("""
    SELECT 
        formatted_work_type AS type_emploi,
        COUNT(*) AS nb_offres
    FROM linkedin.raw.job_postings
    WHERE formatted_work_type IS NOT NULL
    GROUP BY formatted_work_type
    ORDER BY nb_offres DESC
""")
 
df = queried_data.to_pandas()
 
st.subheader("Graphique")
st.bar_chart(data=df, x="TYPE_EMPLOI", y="NB_OFFRES")
 
st.subheader("Données détaillées")
st.dataframe(df[["TYPE_EMPLOI", "NB_OFFRES"]])

Ce code nous ressort ainsi dans streamlit, le visuel suivant :

![Répartition par secteur d'activité](https://raw.githubusercontent.com/clara-cj/Projet-Linkedln-EmmaNEDELEC-Clara-JULIEN/main/Images/Streamlit-5em%20analyses.png)

![Répartition par secteur d'activité - détails](https://raw.githubusercontent.com/clara-cj/Projet-Linkedln-EmmaNEDELEC-Clara-JULIEN/main/Images/Streamlit-5em%20analyses%20details.png)

=> Ici, on remarque que le marché est dominé par des emplois à temps plein (Full-Time) avec 12844 offres, ensuite on retrouve les CDD (Contract) avec 1739 offres et les emplois à temps partiels (Part-Time) représentent 1010 offres. Tandis que les autres catégories sont presque invisibles sur le graphique. Ainsi on a 8 chances sur 10 de tomber sur une offre à temps plein.
 

# III - Problèmes rencontrés et solutions apportées

Un des problèmes que nous avons rencontrés, nottament dans la partie streamlit, à été le fait que les entreprise s'affichaient non pas par leur nom mais par leur numéro d'id inscrit dans les fichiers. Les demandes et les informations n'étaient alors pas réellement claire. Ce problème apparaissait pour les analyses 1, 2 et 4. 

Ce problème n'a été résolut que pour l'analyse 4, où nous sommes ainsi passé de ce premier code : 

In [ ]:
# Premier code éffectué pour l'analyse n°4

conn = st.connection("snowflake")
session = conn.session()
st.title("Analyse 4 : 🏭 Répartition des offres d'emploi par secteur d'activité")
queried_data = session.sql("""
    SELECT
        ji.industry_id AS secteur,
        COUNT(*) AS nb_offres
    FROM linkedin.raw.job_postings jp
    JOIN linkedin.raw.job_industries ji ON jp.job_id = ji.job_id
    WHERE ji.industry_id IS NOT NULL
    GROUP BY ji.industry_id
    ORDER BY nb_offres DESC
""")
df = queried_data.to_pandas()
st.subheader("Graphique")
st.bar_chart(data=df, x="SECTEUR", y="NB_OFFRES")
st.subheader("Données détaillées")
st.dataframe(df[["SECTEUR", "NB_OFFRES"]])

à celui présent dans la partie II de ce projet :

In [ ]:
 # Nouveau code analyse n°4

st.title("Analyse 4 : 🏭 Répartition des offres d'emploi par secteur d'activité")
 
queried_data = session.sql("""
    SELECT
        ci.industry AS secteur,
        COUNT(*) AS nb_offres
    FROM linkedin.raw.job_postings jp
    JOIN linkedin.raw.companies c 
        ON TRY_TO_NUMBER(jp.company_name) = c.company_id
    JOIN linkedin.raw.company_industries ci 
        ON c.company_id = ci.company_id
    WHERE ci.industry IS NOT NULL
    GROUP BY ci.industry
    ORDER BY nb_offres DESC
""")
 
df = queried_data.to_pandas()
 
st.subheader("Graphique")
st.bar_chart(data=df, x="SECTEUR", y="NB_OFFRES")
 
st.subheader("Données détaillées")
st.dataframe(df[["SECTEUR", "NB_OFFRES"]])

=> Nous avons ainsi modifier les termes employés dans le code de 'job_industries' qui ne comporte que les ID, nous avons utilisé 'company_industries' avec les vrais noms. 